# Analisis Data Menggunakan Naive Bayes

## Deskripsi Proyek

Proyek ini digunakan untuk membuat analisis data menggunakan algoritma **Naive Bayes**.

Model classifier Naive Bayes dibuat menggunakan **script Python murni**, tanpa menggunakan Node.js dan tanpa KNIME. Library utama yang digunakan adalah **scikit-learn**, khususnya modul `sklearn.naive_bayes`.

Dataset yang digunakan adalah data film. Fitur teks seperti `title`, `overview`, dan `genre` diproses menggunakan **TF-IDF Vectorizer**, kemudian diklasifikasikan menggunakan algoritma **Multinomial Naive Bayes**.

Tujuan model adalah memprediksi **genre utama film** berdasarkan informasi teks yang tersedia.

## Import Library

Pada tahap ini kita mengimpor library yang dibutuhkan untuk pengolahan data, pembuatan model, dan evaluasi.

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Load Dataset

Dataset film dibaca menggunakan pandas. Dataset ini berisi informasi seperti judul, deskripsi (overview), dan genre.

In [14]:
df = pd.read_csv("n_movies.csv")
df.head()

,title,year,certificate,duration,genre,rating,description,stars,votes
0,Cobra Kai,(2018– ),TV-14,30 min,"Action, Comedy, Drama",8.5,Decades after their 1984 All Valley Karate Tou...,"['Ralph Macchio, ', 'William Zabka, ', 'Courtn...","177,031"
1,The Crown,(2016– ),TV-MA,58 min,"Biography, Drama, History",8.7,Follows the political rivalries and romance of...,"['Claire Foy, ', 'Olivia Colman, ', 'Imelda St...","199,885"
2,Better Call Saul,(2015–2022),TV-MA,46 min,"Crime, Drama",8.9,The trials and tribulations of criminal lawyer...,"['Bob Odenkirk, ', 'Rhea Seehorn, ', 'Jonathan...","501,384"
3,Devil in Ohio,(2022),TV-MA,356 min,"Drama, Horror, Mystery",5.9,When a psychiatrist shelters a mysterious cult...,"['Emily Deschanel, ', 'Sam Jaeger, ', 'Gerardo...","9,773"
4,Cyberpunk: Edgerunners,(2022– ),TV-MA,24 min,"Animation, Action, Adventure",8.6,A Street Kid trying to survive in a technology...,"['Zach Aguilar, ', 'Kenichiro Ohashi, ', 'Emi ...","15,413"


## Eksplorasi Data

Melihat struktur dataset dan mengecek apakah terdapat nilai kosong (missing values).

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9957 entries, 0 to 9956
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        9957 non-null   str    
 1   year         9430 non-null   str    
 2   certificate  6504 non-null   str    
 3   duration     7921 non-null   str    
 4   genre        9884 non-null   str    
 5   rating       8784 non-null   float64
 6   description  9957 non-null   str    
 7   stars        9957 non-null   str    
 8   votes        8784 non-null   str    
dtypes: float64(1), str(8)
memory usage: 700.2 KB


In [16]:
df.isnull().sum()

title             0
year            527
certificate    3453
duration       2036
genre            73
rating         1173
description       0
stars             0
votes          1173
dtype: int64

## Preprocessing Data

Tahap ini bertujuan untuk:
1. Menggabungkan fitur teks menjadi satu kolom (`text`)
2. Mengambil genre utama sebagai target
3. Membersihkan data kosong

In [ ]:
text_columns = ["title", "overview", "genre"]

for col in text_columns:
    if col not in df.columns:
        df[col] = ""

df["text"] = (
    df["title"].fillna("") + " " +
    df["overview"].fillna("") + " " +
    df["genre"].fillna("")
)

df["primary_genre"] = df["genre"].fillna("").apply(
    lambda x: str(x).split(",")[0].strip()
)

df = df[df["primary_genre"] != ""]
df = df[df["text"].str.strip() != ""]

df[["text", "primary_genre"]].head()

Contoh data setelah preprocessing:


,text,primary_genre
0,"Cobra Kai Action, Comedy, Drama",Action
1,"The Crown Biography, Drama, History",Biography
2,"Better Call Saul Crime, Drama",Crime
3,"Devil in Ohio Drama, Horror, Mystery",Drama
4,"Cyberpunk: Edgerunners Animation, Action, Adv...",Animation


## Distribusi Genre

Melihat jumlah data pada setiap genre untuk mendeteksi kelas yang terlalu sedikit.

In [27]:
genre_counts = df["primary_genre"].value_counts()

print("Distribusi genre:")
print(genre_counts)

Distribusi genre:
primary_genre
Comedy         2106
Drama          1723
Animation      1420
Documentary    1345
Action         1220
Crime           670
Adventure       315
Biography       206
Reality-TV      204
Horror          164
Short            95
Thriller         73
Family           65
Game-Show        61
Romance          41
Music            36
Fantasy          32
Mystery          31
Talk-Show        28
Sci-Fi           20
Western           7
Sport             6
News              6
Musical           5
History           4
Name: count, dtype: int64


## Filtering Genre dengan Data Sedikit

Algoritma `train_test_split` dengan `stratify` membutuhkan minimal 2 data per kelas.

Genre yang hanya memiliki 1 data (misalnya Film-Noir) akan dihapus.

In [28]:
valid_genres = genre_counts[genre_counts >= 2].index
df = df[df["primary_genre"].isin(valid_genres)]

print("Jumlah data setelah filtering:", len(df))
print("Jumlah genre:", df["primary_genre"].nunique())

print("\nDistribusi genre setelah filtering:")
print(df["primary_genre"].value_counts())

Jumlah data setelah filtering: 9883
Jumlah genre: 25

Distribusi genre setelah filtering:
primary_genre
Comedy         2106
Drama          1723
Animation      1420
Documentary    1345
Action         1220
Crime           670
Adventure       315
Biography       206
Reality-TV      204
Horror          164
Short            95
Thriller         73
Family           65
Game-Show        61
Romance          41
Music            36
Fantasy          32
Mystery          31
Talk-Show        28
Sci-Fi           20
Western           7
Sport             6
News              6
Musical           5
History           4
Name: count, dtype: int64


## Menentukan Fitur dan Target

In [29]:
X = df["text"]
y = df["primary_genre"]

print("Contoh fitur:")
print(X.head())

print("\nContoh target:")
print(y.head())

Contoh fitur:
0                     Cobra Kai  Action, Comedy, Drama
1                 The Crown  Biography, Drama, History
2                       Better Call Saul  Crime, Drama
3                Devil in Ohio  Drama, Horror, Mystery
4    Cyberpunk: Edgerunners  Animation, Action, Adv...
Name: text, dtype: str

Contoh target:
0       Action
1    Biography
2        Crime
3        Drama
4    Animation
Name: primary_genre, dtype: str


## Split Data Training dan Testing

Dataset dibagi menjadi 80% training dan 20% testing.

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data training:", len(X_train))
print("Jumlah data testing:", len(X_test))

print("\nDistribusi y_train:")
print(y_train.value_counts())

print("\nDistribusi y_test:")
print(y_test.value_counts())

Jumlah data training: 7906
Jumlah data testing: 1977

Distribusi y_train:
primary_genre
Comedy         1685
Drama          1378
Animation      1136
Documentary    1076
Action          976
Crime           536
Adventure       252
Biography       165
Reality-TV      163
Horror          131
Short            76
Thriller         58
Family           52
Game-Show        49
Romance          33
Music            29
Fantasy          25
Mystery          25
Talk-Show        22
Sci-Fi           16
Western           6
News              5
Sport             5
Musical           4
History           3
Name: count, dtype: int64

Distribusi y_test:
primary_genre
Comedy         421
Drama          345
Animation      284
Documentary    269
Action         244
Crime          134
Adventure       63
Reality-TV      41
Biography       41
Horror          33
Short           19
Thriller        15
Family          13
Game-Show       12
Romance          8
Music            7
Fantasy          7
Mystery          6
Talk-Show 

## Pembuatan Model Naive Bayes

In [31]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("nb", MultinomialNB())
])

print(model)

Pipeline(steps=[('tfidf', TfidfVectorizer(stop_words='english')),
                ('nb', MultinomialNB())])


## Training Model

In [33]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('nb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


## Prediksi Data Testing

In [34]:
y_pred = model.predict(X_test)

print("Contoh hasil prediksi:")
print(y_pred[:10])

Contoh hasil prediksi:
['Comedy' 'Comedy' 'Adventure' 'Documentary' 'Comedy' 'Drama' 'Animation'
 'Animation' 'Action' 'Action']


## Evaluasi Model

In [25]:
from sklearn.metrics import accuracy_score, classification_report

print("Akurasi:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Akurasi: 0.7789580171977744
              precision    recall  f1-score   support

      Action       0.77      0.88      0.82       244
   Adventure       1.00      0.43      0.60        63
   Animation       0.84      0.96      0.90       284
   Biography       1.00      0.02      0.05        41
      Comedy       0.71      0.96      0.82       421
       Crime       1.00      0.60      0.75       134
 Documentary       0.81      0.94      0.87       269
       Drama       0.74      0.80      0.77       345
      Family       0.00      0.00      0.00        13
     Fantasy       0.00      0.00      0.00         7
   Game-Show       0.00      0.00      0.00        12
     History       0.00      0.00      0.00         1
      Horror       0.00      0.00      0.00        33
       Music       0.00      0.00      0.00         7
     Musical       0.00      0.00      0.00         1
     Mystery       0.00      0.00      0.00         6
        News       0.00      0.00      0.00         1

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

## perbandingan

In [35]:
# Bandingkan prediksi vs asli
hasil = pd.DataFrame({
    "Teks": X_test.values[:10],
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
})

hasil

,Teks,Actual,Predicted
0,"Celeste & Jesse Forever Comedy, Drama, Romance",Comedy,Comedy
1,"Your Life Is a Joke Documentary, Comedy",Documentary,Comedy
2,"Top Gear Adventure, Comedy, Reality-TV",Adventure,Adventure
3,"Mission Blue Documentary, Drama",Documentary,Documentary
4,Grace and Frankie Comedy,Comedy,Comedy
5,"Oxygen Drama, Fantasy, Sci-Fi",Drama,Drama
6,Untitled Patrick Osborne Fantasy Project Anim...,Animation,Animation
7,"Cry Babies Magic Tears Animation, Family",Animation,Animation
8,"Johan Falk: Organizatsija Karayan Action, Cri...",Action,Action
9,"The Rebel Hong Gil Dong Action, Drama, History",Action,Action
